# Day 1 — 모델 직렬화 실습 (섹션 5)

MNIST 모델을 학습하고 state_dict / TorchScript / ONNX 세 가지로 저장·검증한다.


## 5.2 Step 2 — 모델 학습


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np, os


In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


In [ ]:
BATCH_SIZE, LEARNING_RATE, EPOCHS = 64, 1e-3, 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST('data', train=True, download=True, transform=transform)
test_ds  = datasets.MNIST('data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
model = SimpleClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [ ]:
for epoch in range(1, EPOCHS+1):
    model.train(); running, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(images); loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        running += loss.item(); total += labels.size(0)
        correct += out.argmax(1).eq(labels).sum().item()
    print(f'Epoch {epoch}/{EPOCHS} Loss={running/len(train_loader):.4f} Acc={100*correct/total:.1f}%')


## 5.3 Step 3 — 세 가지 방식으로 저장


In [ ]:
os.makedirs('models', exist_ok=True)
model_cpu = model.cpu().eval()
test_input = test_ds[0][0].unsqueeze(0)
test_label = test_ds[0][1]
with torch.no_grad():
    original_output = model_cpu(test_input)
    original_pred = original_output.argmax(1).item()

torch.save(model_cpu.state_dict(), 'models/mnist_state_dict.pth')
torch.jit.trace(model_cpu, test_input).save('models/mnist_traced.pt')
torch.onnx.export(model_cpu, test_input, 'models/mnist_model.onnx',
                  export_params=True, opset_version=17,
                  input_names=['image'], output_names=['prediction'],
                  dynamic_axes={'image': {0: 'batch_size'}, 'prediction': {0: 'batch_size'}})
print('저장 완료:', sorted(os.listdir('models')))


## 5.4 Step 4 — 불러오기 및 추론 검증


In [ ]:
loaded_sd = SimpleClassifier()
loaded_sd.load_state_dict(torch.load('models/mnist_state_dict.pth', weights_only=True)); loaded_sd.eval()
loaded_ts = torch.jit.load('models/mnist_traced.pt')
import onnxruntime as ort
session = ort.InferenceSession('models/mnist_model.onnx')
with torch.no_grad():
    sd_pred = loaded_sd(test_input).argmax(1).item()
    ts_pred = loaded_ts(test_input).argmax(1).item()
onnx_pred = int(np.argmax(session.run(['prediction'], {'image': test_input.numpy()})[0], axis=1)[0])
print('정답', test_label, '| state_dict', sd_pred, '| TorchScript', ts_pred, '| ONNX', onnx_pred)


## 5.5 Step 5 — 배치 추론


In [ ]:
batch = torch.stack([test_ds[i][0] for i in range(8)])
labels = [test_ds[i][1] for i in range(8)]
with torch.no_grad():
    sd_b = loaded_sd(batch).argmax(1).tolist()
onnx_b = np.argmax(session.run(['prediction'], {'image': batch.numpy()})[0], axis=1).tolist()
print('정답   :', labels)
print('예측   :', sd_b)
print('ONNX   :', onnx_b)


## 5.6 Step 6 — 추론 함수 분리 테스트


In [ ]:
import sys; sys.path.insert(0, '.')
from app.model_utils import load_model, predict
m = load_model('models/mnist_state_dict.pth')
print(predict(m, test_input))
